# Forecast experiment

This notebook optimizes initial conditions (x_0) using manual gradient descent.
Visualization and metrics at each step can help to understand better. </br>
From a series of observation (fully-observed), an initial condition can be optmized.
This notebook aims to see the optimal perturbation can lead to better forecast. 

*[!]This notebook script sees only surface ocean states*


## 1. Setup and Imports

In [ ]:
import torch
import torch.nn as nn
import xarray as xr
import numpy as np
import os
import sys
from pathlib import Path
from datetime import datetime
from typing import Tuple, Dict
import gc
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from IPython.display import clear_output, display
import pandas as pd

# Add paths
sys.path.append(str(Path.cwd().parent.parent / "glonet_daily_forecast_local"))
sys.path.append(str(Path.cwd().parent.parent / "src/glonet"))
sys.path.append(str(Path.cwd().parent / "src"))
 
from utility import get_normalizer1 #, get_normalizer2, get_normalizer3
from utility import get_denormalizer1 #, get_denormalizer2, get_denormalizer3
from modelp2 import Glonet
from optimIC_GD_glonetLit import GlonetGradientCheckpointing

# Set matplotlib style
%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')

print("✓ All imports successful")

## 2. Configuration

In [ ]:
# Configuration parameters
CONFIG = {
    'data_path': "/Odyssey/public/glonet/glorys12_2020-01-01_to_2020-01-31_init_states/combined_input.nc", 
    'model_location': "/Odyssey/public/glonet/TrainedWeights",
    'learning_rate': 1e-1,
    'num_iterations': 1000,
    'sample_idx': 2,
    'sequence_length': 2,
    'observation_length': 7,
    'forecast_horizon': 14,
    'device': "cuda:0" if torch.cuda.is_available() else "cpu",
    'output_dir': f"../outputs/forecast_exp/obs_7d-forec_14d_gradf_idx2_{datetime.now().strftime('%Y%m%d_%H%M%S')}",
    'plot_frequency': 5,  # Plot every N iterations
    'gradient_filter': 'pooling',  # Options: 'none', 'weight', 'delta', 'pooling', 'pooling+weight'
    'pooling_kernel': 12,  # Kernel size for average pooling (must divide evenly into spatial dimensions)
    
    # Multigrid / Scheduled Pooling Configuration
    'use_scheduled_pooling': True,  # Enable scheduled pooling
    'pooling_schedule': {
        'type': 'step',  # Options: 'step', 'linear', 'exponential'
        'initial_kernel': 24,  # Starting kernel size (coarse)
        'final_kernel': 1,     # Ending kernel size (fine/no pooling)
        'schedule_steps': [100, 200, 400, 600, 800],  # Iterations to change kernel size (for step schedule)
        'kernel_sizes': [8, 4, 8, 2, 4, 2],  # Corresponding kernel sizes at each step
    }
}

# Create output directory
os.makedirs(CONFIG['output_dir'], exist_ok=True)

print("Configuration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

### Multigrid / Scheduled Pooling Options

The notebook now supports **scheduled pooling** (multigrid optimization), where the pooling kernel size changes during optimization:

**Benefits:**
- Start with **coarse pooling** (large kernel) for global optimization
- Gradually reduce to **fine pooling** (small kernel) for local refinement
- Avoids getting stuck in local minima early in optimization

**Schedule Types:**

1. **Step Schedule** (recommended): Changes kernel at specific iterations
   ```python
   'pooling_schedule': {
       'type': 'step',
       'schedule_steps': [100, 200, 300, 400],
       'kernel_sizes': [24, 12, 6, 3, 1],
   }
   ```

2. **Linear Schedule**: Linearly interpolates from initial to final kernel
   ```python
   'pooling_schedule': {
       'type': 'linear',
       'initial_kernel': 24,
       'final_kernel': 1,
   }
   ```

3. **Exponential Schedule**: Exponentially decays from initial to final kernel
   ```python
   'pooling_schedule': {
       'type': 'exponential',
       'initial_kernel': 24,
       'final_kernel': 1,
   }
   ```

**To disable scheduled pooling**, set `'use_scheduled_pooling': False` and use fixed `'pooling_kernel'`.

**Note:** Kernel sizes should divide evenly into spatial dimensions (672, 1440). Valid kernels include: 1, 2, 3, 4, 6, 8, 12, 16, 24, 48, etc.

## 3. Load Models and Normalizers

In [ ]:
print("Loading normalizers and denormalizers...")
normalizer1 = get_normalizer1(CONFIG['model_location'])
# normalizer2 = get_normalizer2(CONFIG['model_location'])
# normalizer3 = get_normalizer3(CONFIG['model_location'])

denormalizer1 = get_denormalizer1(CONFIG['model_location'])
# denormalizer2 = get_denormalizer2(CONFIG['model_location'])
# denormalizer3 = get_denormalizer3(CONFIG['model_location'])

print("✓ Normalizers loaded")

In [ ]:
def load_checkpoint_model(checkpoint_path: str, shape_in: Tuple, device: str):
    """Load a checkpoint-based model with gradient checkpointing."""
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model = GlonetGradientCheckpointing(shape_in=shape_in)
    model.load_state_dict(checkpoint['model_state_dict'])
    model = model.to(device)
    model.train()  # Enable gradient checkpointing
    for param in model.parameters():
        param.requires_grad = False
    return model

print("Loading pretrained models...")
model1 = load_checkpoint_model(
    f"{CONFIG['model_location']}/glonet_part1.pth",
    shape_in=(2, 5, 672, 1440),
    device=CONFIG['device']
)
# model2 = load_checkpoint_model(
#     f"{CONFIG['model_location']}/glonet_part2.pth",
#     shape_in=(2, 40, 672, 1440),
#     device=CONFIG['device']
# )
# model3 = load_checkpoint_model(
#     f"{CONFIG['model_location']}/glonet_part3.pth",
#     shape_in=(2, 40, 672, 1440),
#     device=CONFIG['device']
# )

loss_fn = nn.MSELoss()

print("✓ Models loaded successfully")

## 4. Load and Prepare Data

In [ ]:
print(f"Loading dataset from {CONFIG['data_path']}...")
dataset = xr.open_dataset(CONFIG['data_path'])

# Extract input sequence and forecast target sequence
input_sequence = dataset.isel(
    time=slice(CONFIG['sample_idx'], CONFIG['sample_idx'] + CONFIG['sequence_length'])
)
target_start_idx = CONFIG['sample_idx'] + CONFIG['sequence_length']
target_end_idx = target_start_idx + CONFIG['observation_length']
target_sequence = dataset.isel(time=slice(target_start_idx, target_end_idx))
target_time_idx = target_end_idx - 1
target = dataset.isel(time=target_time_idx)

# Store coordinates
coords = {
    'time': input_sequence['time'].values,
    'target_times': target_sequence['time'].values,
    'target_time': target['time'].values,
    'lat': input_sequence['lat'].values,
    'lon': input_sequence['lon'].values
}

print(f"Input sequence shape: {input_sequence['data'].shape}")
print(f"Target sequence shape: {target_sequence['data'].shape}")
print(f"Final target shape: {target['data'].shape}")
print(f"Final target time: {coords['target_time']}")

In [ ]:
# # Uncertainty for initial condition and observations
# # e ~ N(0, var_channel), with a tunable multiplier
# noise_scale = 1.0
# noise_seed = 42
# rng = np.random.default_rng(noise_seed)

# # Define funciton
# def add_channelwise_gaussian_noise(ds, var_name='data', scale=1.0):
#     da = ds[var_name]

#     # Variance by channel over time/space
#     var_by_channel = da.var(dim=('time', 'lat', 'lon'), skipna=True)
#     std_by_channel = np.sqrt(var_by_channel.values + 1e-12) * scale

#     noise = rng.normal(
#         loc=0.0,
#         scale=std_by_channel[None, :, None, None],
#         size=da.shape
#     )

#     ds_noisy = ds.copy()
#     ds_noisy[var_name] = da + xr.DataArray(
#         noise,
#         dims=da.dims,
#         coords=da.coords
#     )

#     return ds_noisy, var_by_channel

# # Noisy states
# input_sequence, input_var_by_channel = add_channelwise_gaussian_noise(
#     input_sequence,
#     scale=noise_scale
# )

# target_sequence, target_var_by_channel = add_channelwise_gaussian_noise(
#     target_sequence,
#     scale=noise_scale
# )

# print('Added channel-wise Gaussian noise: e ~ N(0, var_channel)')
# print(f'Noise scale: {noise_scale}, seed: {noise_seed}')
# print('Input variance by channel:')
# print(input_var_by_channel.values)
# print('Target variance by channel:')
# print(target_var_by_channel.values)

In [ ]:
# Create ocean masks
sample_data = dataset.isel(time=CONFIG['sample_idx'])
data_values = sample_data['data'].values
land_mask = np.isnan(data_values).astype(np.float32)

ocean_mask_1 = torch.from_numpy((1.0 - land_mask[0:5, :, :]).copy()).float().to(CONFIG['device'])
# ocean_mask_2 = torch.from_numpy((1.0 - land_mask[5:45, :, :]).copy()).float().to(CONFIG['device'])
# ocean_mask_3 = torch.from_numpy((1.0 - land_mask[45:85, :, :]).copy()).float().to(CONFIG['device'])
ocean_mask_all = ocean_mask_1
# ocean_mask_all = torch.cat([ocean_mask_1, ocean_mask_2, ocean_mask_3], dim=0)

print(f"Ocean mask 1: {ocean_mask_all.shape}")
# print(f"Ocean mask 2: {ocean_mask_2.shape}")
# print(f"Ocean mask 3: {ocean_mask_3.shape}")

In [ ]:
# Prepare initial conditions and target sequence
input_data = input_sequence['data'].values
target_sequence_data = target_sequence['data'].values

input_data = np.nan_to_num(input_data, nan=0.0)
target_sequence_data = np.nan_to_num(target_sequence_data, nan=0.0)

x0 = torch.from_numpy(input_data[:, 0:5, :, :].copy()).float().unsqueeze(0).to(CONFIG['device'])
x0_ref = x0.clone()
target_tensor_seq = torch.from_numpy(target_sequence_data[:, 0:5, :, :].copy()).float().unsqueeze(0).to(CONFIG['device'])
target_tensor = target_tensor_seq[:, -1, :, :, :]

x0.requires_grad = True
x0_ref.requires_grad = False

print(f"Initial condition shape: {x0.shape}")
print(f"Target sequence shape: {target_tensor_seq.shape}")
print(f"Final target shape: {target_tensor.shape}")
print("✓ Data preparation complete")

## 5. Define Functions

In [ ]:
def get_pooling_kernel_size(iteration, config):
    """Compute the pooling kernel size for the current iteration based on schedule.
    
    Args:
        iteration: Current iteration number (0-indexed)
        config: Configuration dictionary
    
    Returns:
        kernel_size: Integer kernel size to use for pooling
    """
    if not config.get('use_scheduled_pooling', False):
        # Use fixed kernel size if scheduled pooling is disabled
        return config['pooling_kernel']
    
    schedule = config['pooling_schedule']
    schedule_type = schedule['type']
    
    if schedule_type == 'step':
        # Step-wise schedule: change kernel at specific iterations
        steps = schedule['schedule_steps']
        kernels = schedule['kernel_sizes']
        
        # Find which step we're in
        kernel_idx = 0
        for i, step in enumerate(steps):
            if iteration >= step:
                kernel_idx = i + 1
        
        return kernels[kernel_idx]
    
    elif schedule_type == 'linear':
        # Linear interpolation from initial to final kernel
        initial = schedule['initial_kernel']
        final = schedule['final_kernel']
        total_iters = config['num_iterations']
        
        # Linear decay
        progress = min(iteration / total_iters, 1.0)
        kernel_size = int(initial - (initial - final) * progress)
        kernel_size = max(kernel_size, final)  # Don't go below final
        
        return kernel_size
    
    elif schedule_type == 'exponential':
        # Exponential decay from initial to final kernel
        initial = schedule['initial_kernel']
        final = schedule['final_kernel']
        total_iters = config['num_iterations']
        
        # Exponential decay
        progress = min(iteration / total_iters, 1.0)
        kernel_size = int(initial * ((final / initial) ** progress))
        kernel_size = max(kernel_size, final)  # Don't go below final
        
        return kernel_size
    
    else:
        raise ValueError(f"Unknown schedule type: {schedule_type}")

print("✓ Pooling kernel schedule function defined")

In [ ]:
def forward(x, 
            forward_iteration: int = 1):
    """Forward pass through the model and return predictions for all forecast steps."""
    
    with torch.enable_grad():
        x1 = x[:, :, 0:5, :, :]
        x1_in_std = normalizer1(x1) * ocean_mask_1.unsqueeze(0).unsqueeze(0)
        # x2 = x[:, :, 5:45, :, :]
        # x2_in_std = normalizer2(x2) * ocean_mask_2.unsqueeze(0).unsqueeze(0)
        # x3 = x[:, :, 45:85, :, :]
        # x3_in_std = normalizer3(x3) * ocean_mask_3.unsqueeze(0).unsqueeze(0)
        
        y_hat1 = model1(x1_in_std)
        # y_hat2 = model2(x2_in_std)
        # y_hat3 = model3(x3_in_std)
        y_hat_steps = [denormalizer1(y_hat1[:, -1, :, :, :])]
            
        # Auto regressive forecasting for forecast_horizon steps
        # Following forecast.py: output goes directly back as input
        if forward_iteration > 1:
            for i in range(forward_iteration - 1):
                y_hat1 = model1(y_hat1)
                # y_hat2 = model2(y_hat2)
                # y_hat3 = model3(y_hat3)
                y_hat_steps.append(denormalizer1(y_hat1[:, -1, :, :, :]))
    
    y_hat_steps = torch.stack(y_hat_steps, dim=1)
    # y_hat = torch.cat([y_hat1, y_hat2, y_hat3], dim=1)
    # x0_in_std = torch.cat([x1_in_std, x2_in_std, x3_in_std], dim=2)
    return x1_in_std, y_hat_steps

def compute_loss(y_hat_steps):
    """Compute per-step losses and total loss as the sum over forecast steps."""
    
    # target_ssh = target_tensor_seq[:, :, 0:1, :, :]
    # target_t = torch.cat([target_tensor_seq[:, :, 1:2, :, :], target_tensor_seq[:, :, 5:15, :, :]], dim=2)
    # target_s = torch.cat([target_tensor_seq[:, :, 2:3, :, :], target_tensor_seq[:, :, 15:25, :, :]], dim=2)
    # target_u = torch.cat([target_tensor_seq[:, :, 3:4, :, :], target_tensor_seq[:, :, 25:35, :, :]], dim=2)
    # target_v = torch.cat([target_tensor_seq[:, :, 4:5, :, :], target_tensor_seq[:, :, 35:45, :, :]], dim=2)
    
    # pred_ssh = y_hat_steps[:, :, 0:1, :, :]
    # pred_t = torch.cat([y_hat_steps[:, :, 1:2, :, :], y_hat_steps[:, :, 5:15, :, :]], dim=2)
    # pred_s = torch.cat([y_hat_steps[:, :, 2:3, :, :], y_hat_steps[:, :, 15:25, :, :]], dim=2)
    # pred_u = torch.cat([y_hat_steps[:, :, 3:4, :, :], y_hat_steps[:, :, 25:35, :, :]], dim=2)
    # pred_v = torch.cat([y_hat_steps[:, :, 4:5, :, :], y_hat_steps[:, :, 35:45, :, :]], dim=2)
    
    se = ((target_tensor_seq - y_hat_steps) ** 2)                           # [1, T, 5, H, W]
    se_masked = se * ocean_mask_all.unsqueeze(0).unsqueeze(0)               # [1, T, 5, H, W]
    n_ocean_points = ocean_mask_all.unsqueeze(0).sum(dim=(2, 3)).unsqueeze(1)  # [1, 1, 5]
    mse_per_step = se_masked.sum(dim=(3, 4)) / (n_ocean_points + 1e-10)     # [1, T, 5]
    
    var = target_tensor_seq.var(dim=(3, 4))                                 # [1, T, 5]
    nmse_per_step = mse_per_step / (var + 1e-10)                             # [1, T, 5]
    
    # Sum losses over time for each channel, then over channels for total objective
    total_nloss = nmse_per_step.sum(dim=1)                                   # [1, 5]
    total_loss = total_nloss.sum(dim=1)                                      # [1]
    
    return mse_per_step, nmse_per_step, total_nloss, total_loss

print("✓ Forward and loss functions defined")

In [ ]:
def apply_gradient_filter(gradient, x0, filter_type='none', pooling_kernel=None):
    """Apply different filtering strategies to gradients.
    
    Args:
        gradient: Raw gradient tensor [1, T, 85, H, W]
        x0: Current initial condition [1, T, 85, H, W]
        filter_type: Type of filter to apply
            - 'none': No filtering (return raw gradient)
            - 'weight': Weight by variable variance
            - 'delta': Adaptive step size based on gradient magnitude
            - 'pooling': Average pooling + upsampling for global optimization
            - 'pooling+weight': Combination of pooling and variance weighting
        pooling_kernel: Kernel size for pooling operations (overrides CONFIG if provided)
    
    Returns:
        filtered_gradient: Filtered gradient tensor [1, T, 85, H, W]
    """
    filtered_grad = gradient.clone()
    
    if filter_type == 'none':
        return filtered_grad
    
    elif filter_type == 'weight':
        # Weight gradients by per-channel variance (dynamic over all channels)
        with torch.no_grad():
            channel_vars = x0[:, 0].var(dim=(0, 2, 3), keepdim=False) + 1e-10  # [C]
            weight = channel_vars.view(1, 1, -1, 1, 1)  # [1, 1, C, 1, 1]
            filtered_grad = filtered_grad * weight
    
    elif filter_type == 'delta':
        # Adaptive filtering: normalize by gradient magnitude per channel
        with torch.no_grad():
            for t_idx in range(filtered_grad.shape[1]):
                for ch_idx in range(filtered_grad.shape[2]):
                    grad_ch = filtered_grad[0, t_idx, ch_idx]
                    ocean_points = grad_ch[ocean_mask_all[ch_idx] > 0]
                    if len(ocean_points) > 0:
                        grad_norm = torch.norm(ocean_points)
                        if grad_norm > 1e-10:
                            filtered_grad[0, t_idx, ch_idx] = grad_ch / (grad_norm + 1e-10)
    
    elif filter_type == 'pooling':
        # Average pooling + upsampling for global optimization
        # Use provided kernel size or fall back to CONFIG
        kernel_size = pooling_kernel if pooling_kernel is not None else CONFIG['pooling_kernel']
        
        with torch.no_grad():
            # Apply pooling and upsampling per channel
            for t_idx in range(filtered_grad.shape[1]):
                for ch_idx in range(filtered_grad.shape[2]):
                    grad_ch = filtered_grad[0, t_idx, ch_idx].unsqueeze(0).unsqueeze(0)  # [1, 1, H, W]
                    
                    # Average pooling
                    pooled = nn.functional.avg_pool2d(grad_ch, kernel_size=kernel_size, stride=kernel_size)
                    
                    # Upsample back to original size
                    upsampled = nn.functional.interpolate(
                        pooled, 
                        size=(grad_ch.shape[2], grad_ch.shape[3]), 
                        mode='bilinear', 
                        align_corners=False
                    )
                    
                    filtered_grad[0, t_idx, ch_idx] = upsampled.squeeze(0).squeeze(0)
                    
                    # Re-apply ocean mask
                    filtered_grad[0, t_idx, ch_idx] *= ocean_mask_all[ch_idx]
    
    elif filter_type == 'pooling+weight':
        # Combination: first apply pooling, then variance weighting
        # Use provided kernel size or fall back to CONFIG
        kernel_size = pooling_kernel if pooling_kernel is not None else CONFIG['pooling_kernel']
        
        with torch.no_grad():
            # Step 1: Pooling
            for t_idx in range(filtered_grad.shape[1]):
                for ch_idx in range(filtered_grad.shape[2]):
                    grad_ch = filtered_grad[0, t_idx, ch_idx].unsqueeze(0).unsqueeze(0)
                    pooled = nn.functional.avg_pool2d(grad_ch, kernel_size=kernel_size, stride=kernel_size)
                    upsampled = nn.functional.interpolate(
                        pooled, 
                        size=(grad_ch.shape[2], grad_ch.shape[3]), 
                        mode='bilinear', 
                        align_corners=False
                    )
                    filtered_grad[0, t_idx, ch_idx] = upsampled.squeeze(0).squeeze(0)
                    filtered_grad[0, t_idx, ch_idx] *= ocean_mask_all[ch_idx]
            
            # Step 2: Variance weighting (dynamic over all channels)
            channel_vars = x0[:, 0].var(dim=(0, 2, 3), keepdim=False) + 1e-10  # [C]
            weight = channel_vars.view(1, 1, -1, 1, 1)  # [1, 1, C, 1, 1]
            filtered_grad = filtered_grad * weight
    
    else:
        raise ValueError(f"Unknown filter_type: {filter_type}")
    
    return filtered_grad

print("✓ Gradient filtering function defined")

In [ ]:
def compute_rmse_metrics(y_hat, x0_current, x0_reference):
    """Compute RMSE metrics for predictions and initial conditions."""
    with torch.no_grad():
        # Prediction errors
        squared_errors = (target_tensor - y_hat) ** 2
        squared_errors_masked = squared_errors * ocean_mask_all.unsqueeze(0)
        n_ocean_points = ocean_mask_all.unsqueeze(0).sum(dim=(2, 3))
        mse_per_channel = squared_errors_masked.sum(dim=(2, 3)) / (n_ocean_points + 1e-10)
        rmse_per_channel = torch.sqrt(mse_per_channel).cpu().numpy().squeeze(0)
        
        # IC errors
        ic_squared_errors = (x0_reference - x0_current) ** 2
        ic_squared_errors_masked = ic_squared_errors * ocean_mask_all.unsqueeze(0).unsqueeze(0)
        ic_mse_per_channel = ic_squared_errors_masked.sum(dim=(3, 4)) / (n_ocean_points + 1e-10)
        ic_rmse_per_channel = torch.sqrt(ic_mse_per_channel).cpu().numpy().squeeze(0)
        
        # Compute per-variable metrics
        metrics = {
            'pred': {
                'SSH': float(rmse_per_channel[0]),
                'thetao': [float(rmse_per_channel[1:2].mean()), 
                               float(rmse_per_channel[5:15].mean()), 
                               float(rmse_per_channel[45:55].mean())],
                'so': [float(rmse_per_channel[2:3].mean()), 
                           float(rmse_per_channel[15:25].mean()), 
                           float(rmse_per_channel[55:65].mean())],
                'uo': [float(rmse_per_channel[3:4].mean()), 
                           float(rmse_per_channel[25:35].mean()), 
                           float(rmse_per_channel[65:75].mean())],
                'vo': [float(rmse_per_channel[4:5].mean()), 
                           float(rmse_per_channel[35:45].mean()), 
                           float(rmse_per_channel[75:85].mean())],
            },
            'ic': {
                'SSH': float(ic_rmse_per_channel[1, 0]),
                'thetao': [float(ic_rmse_per_channel[1, 1:2].mean()), 
                              float(ic_rmse_per_channel[1, 5:15].mean()), 
                              float(ic_rmse_per_channel[1, 45:55].mean())],
                'so': [float(ic_rmse_per_channel[1, 2:3].mean()), 
                           float(ic_rmse_per_channel[1, 15:25].mean()), 
                           float(ic_rmse_per_channel[1, 55:65].mean())],
                'uo': [float(ic_rmse_per_channel[1, 3:4].mean()), 
                           float(ic_rmse_per_channel[1, 25:35].mean()), 
                           float(ic_rmse_per_channel[1, 65:75].mean())],
                'vo': [float(ic_rmse_per_channel[1, 4:5].mean()), 
                           float(ic_rmse_per_channel[1, 35:45].mean()), 
                           float(ic_rmse_per_channel[1, 75:85].mean())],
            }
        }
        
        return metrics

print("✓ RMSE computation function defined")

## 6. Visualization Functions

In [ ]:
def plot_rmse_evolution(history, iteration):
    """Plot RMSE evolution over iterations with separate subplots for each depth level."""
    # Create figure with 3 columns (Prediction RMSE, IC RMSE, Loss) and multiple rows
    fig, axes = plt.subplots(14, 3, figsize=(24, 36))
    fig.suptitle(f'RMSE and Loss Evolution - Iteration {iteration}', fontsize=16, fontweight='bold')
    
    variables = ['SSH', 'thetao', 'so', 'uo', 'vo']
    var_titles = ['SSH (Sea Surface Height)', 'Temperature (thetao)', 'Salinity (so)', 
                  'Eastward Velocity (uo)', 'Northward Velocity (vo)']
    loss_keys = ['loss_ssh', 'loss_t', 'loss_s', 'loss_u', 'loss_v']
    depths = ['surface', 'shallow', 'deep']
    
    row_idx = 0
    
    for var, title, loss_key in zip(variables, var_titles, loss_keys):
        pred_rmse = [h['pred'][var] for h in history]
        ic_rmse = [h['ic'][var] for h in history]
        var_loss = [h[loss_key] for h in history]
        
        if isinstance(pred_rmse[0], list):
            # Variable with depth levels
            for depth_idx, depth in enumerate(depths):
                pred_rmse_depth = [p[depth_idx] for p in pred_rmse]
                ic_rmse_depth = [ic[depth_idx] for ic in ic_rmse]
                
                # Prediction RMSE subplot
                axes[row_idx, 0].plot(pred_rmse_depth, 'b-', linewidth=2)
                axes[row_idx, 0].set_xlabel('Iteration', fontsize=10)
                axes[row_idx, 0].set_ylabel('RMSE', fontsize=10)
                axes[row_idx, 0].set_title(f'{title} - {depth} (Pred)', fontsize=11, fontweight='bold')
                axes[row_idx, 0].grid(True, alpha=0.3)
                
                # IC RMSE subplot
                axes[row_idx, 1].plot(ic_rmse_depth, 'r-', linewidth=2)
                axes[row_idx, 1].set_xlabel('Iteration', fontsize=10)
                axes[row_idx, 1].set_ylabel('RMSE', fontsize=10)
                axes[row_idx, 1].set_title(f'{title} - {depth} (IC)', fontsize=11, fontweight='bold')
                axes[row_idx, 1].grid(True, alpha=0.3)
                
                # Loss subplot (only for first depth level of each variable)
                if depth_idx == 0:
                    axes[row_idx, 2].plot(var_loss, 'g-', linewidth=2)
                    axes[row_idx, 2].set_xlabel('Iteration', fontsize=10)
                    axes[row_idx, 2].set_ylabel('Loss', fontsize=10)
                    axes[row_idx, 2].set_title(f'{title} Loss', fontsize=11, fontweight='bold')
                    axes[row_idx, 2].grid(True, alpha=0.3)
                else:
                    # Hide loss subplot for non-first depth levels
                    axes[row_idx, 2].axis('off')
                
                row_idx += 1
        else:
            # SSH (no depth levels)
            # Prediction RMSE subplot
            axes[row_idx, 0].plot(pred_rmse, 'b-', linewidth=2)
            axes[row_idx, 0].set_xlabel('Iteration', fontsize=10)
            axes[row_idx, 0].set_ylabel('RMSE', fontsize=10)
            axes[row_idx, 0].set_title(f'{title} (Pred)', fontsize=11, fontweight='bold')
            axes[row_idx, 0].grid(True, alpha=0.3)
            
            # IC RMSE subplot
            axes[row_idx, 1].plot(ic_rmse, 'r-', linewidth=2)
            axes[row_idx, 1].set_xlabel('Iteration', fontsize=10)
            axes[row_idx, 1].set_ylabel('RMSE', fontsize=10)
            axes[row_idx, 1].set_title(f'{title} (IC)', fontsize=11, fontweight='bold')
            axes[row_idx, 1].grid(True, alpha=0.3)
            
            # Loss subplot
            axes[row_idx, 2].plot(var_loss, 'g-', linewidth=2)
            axes[row_idx, 2].set_xlabel('Iteration', fontsize=10)
            axes[row_idx, 2].set_ylabel('Loss', fontsize=10)
            axes[row_idx, 2].set_title(f'{title} Loss', fontsize=11, fontweight='bold')
            axes[row_idx, 2].grid(True, alpha=0.3)
            
            row_idx += 1
    
    # Total Loss plot (spanning all columns in the last row)
    fig.delaxes(axes[row_idx, 1])
    fig.delaxes(axes[row_idx, 2])
    axes[row_idx, 0].set_position([0.125, 0.025, 0.775, 0.055])
    losses = [h['loss'] for h in history]
    axes[row_idx, 0].plot(losses, 'purple', linewidth=2.5)
    axes[row_idx, 0].set_xlabel('Iteration', fontsize=11)
    axes[row_idx, 0].set_ylabel('Total Loss', fontsize=11)
    axes[row_idx, 0].set_title('Total Loss (Sum of All Variables)', fontsize=12, fontweight='bold')
    axes[row_idx, 0].grid(True, alpha=0.3)
    
    plt.tight_layout(rect=[0, 0, 1, 0.98])
    return fig

print("✓ RMSE evolution plot function defined")

In [ ]:
def plot_predictions(y_hat, y_hat_init, x0_current, x0_reference, coords, iteration, channel_idx=0, time_idx=1):
    """Plot initial vs optimized conditions and predictions in a 2x3 grid.
    
    Args:
        y_hat: Current prediction tensor [1, 85, H, W]
        y_hat_init: Initial prediction tensor [1, 85, H, W]
        x0_current: Current initial condition [1, T, 85, H, W]
        x0_reference: Reference initial condition [1, T, 85, H, W]
        coords: Dictionary with lat/lon coordinates
        iteration: Current iteration number
        channel_idx: Channel index to visualize
        time_idx: Time step index for initial conditions (default=1, last time step)
    """
    with torch.no_grad():
        # Predictions
        pred_opt = y_hat[0, channel_idx].cpu().numpy()
        pred_init = y_hat_init[0, channel_idx].cpu().numpy()
        pred_diff = pred_opt - pred_init
        
        # Initial conditions
        ic_opt = x0_current[0, time_idx, channel_idx].cpu().numpy()
        ic_init = x0_reference[0, time_idx, channel_idx].cpu().numpy()
        ic_diff = ic_opt - ic_init
    
    # Replace zeros with NaN for ocean-only visualization
    # pred_opt = np.where(pred_opt == 0, np.nan, pred_opt)
    # pred_init = np.where(pred_init == 0, np.nan, pred_init)
    # pred_diff = np.where(np.isnan(pred_opt) | np.isnan(pred_init), np.nan, pred_diff)
    
    # ic_opt = np.where(ic_opt == 0, np.nan, ic_opt)
    # ic_init = np.where(ic_init == 0, np.nan, ic_init)
    # ic_diff = np.where(np.isnan(ic_opt) | np.isnan(ic_init), np.nan, ic_diff)
    
    land_nan = np.where(ocean_mask_all[channel_idx].cpu().numpy() == 0, np.nan, 1)
    pred_opt = pred_opt * land_nan
    pred_init = pred_init * land_nan
    pred_diff = pred_diff * land_nan
    
    ic_opt = ic_opt * land_nan
    ic_init = ic_init * land_nan
    ic_diff = ic_diff * land_nan
    
    # Get lat/lon coordinates
    lons = coords['lon']
    lats = coords['lat']
    
    fig, axes = plt.subplots(2, 3, figsize=(22, 12))
    fig.suptitle(f'Channel {channel_idx} - Iteration {iteration}', fontsize=16, fontweight='bold')
    
    # Row 1: Initial Conditions
    # Initial IC
    im00 = axes[0, 0].pcolormesh(lons, lats, ic_init, cmap='viridis', shading='auto')
    axes[0, 0].set_title('Initial Condition (Reference)', fontsize=12)
    axes[0, 0].set_xlabel('Longitude (°E)', fontsize=11)
    axes[0, 0].set_ylabel('Latitude (°N)', fontsize=11)
    axes[0, 0].set_aspect('equal')
    plt.colorbar(im00, ax=axes[0, 0], label='Value', fraction=0.025, pad=0.04)
    
    # Optimized IC
    im01 = axes[0, 1].pcolormesh(lons, lats, ic_opt, cmap='viridis', shading='auto')
    axes[0, 1].set_title('Optimized Initial Condition', fontsize=12)
    axes[0, 1].set_xlabel('Longitude (°E)', fontsize=11)
    axes[0, 1].set_ylabel('Latitude (°N)', fontsize=11)
    axes[0, 1].set_aspect('equal')
    plt.colorbar(im01, ax=axes[0, 1], label='Value', fraction=0.025, pad=0.04)
    
    # IC Difference
    max_abs_ic_diff = np.nanmax(np.abs(ic_diff))
    im02 = axes[0, 2].pcolormesh(lons, lats, ic_diff, cmap='RdBu_r', shading='auto',
                                 vmin=-max_abs_ic_diff, vmax=max_abs_ic_diff)
    axes[0, 2].set_title('IC Difference (Opt - Init)', fontsize=12)
    axes[0, 2].set_xlabel('Longitude (°E)', fontsize=11)
    axes[0, 2].set_ylabel('Latitude (°N)', fontsize=11)
    axes[0, 2].set_aspect('equal')
    plt.colorbar(im02, ax=axes[0, 2], label='Difference', fraction=0.025, pad=0.04)
    
    # Row 2: Predictions
    # Initial Prediction
    im10 = axes[1, 0].pcolormesh(lons, lats, pred_init, cmap='viridis', shading='auto')
    axes[1, 0].set_title('Initial Prediction', fontsize=12)
    axes[1, 0].set_xlabel('Longitude (°E)', fontsize=11)
    axes[1, 0].set_ylabel('Latitude (°N)', fontsize=11)
    axes[1, 0].set_aspect('equal')
    plt.colorbar(im10, ax=axes[1, 0], label='Value', fraction=0.025, pad=0.04)
    
    # Optimized Prediction
    im11 = axes[1, 1].pcolormesh(lons, lats, pred_opt, cmap='viridis', shading='auto')
    axes[1, 1].set_title('Optimized Prediction', fontsize=12)
    axes[1, 1].set_xlabel('Longitude (°E)', fontsize=11)
    axes[1, 1].set_ylabel('Latitude (°N)', fontsize=11)
    axes[1, 1].set_aspect('equal')
    plt.colorbar(im11, ax=axes[1, 1], label='Value', fraction=0.025, pad=0.04)
    
    # Prediction Difference
    max_abs_pred_diff = np.nanmax(np.abs(pred_diff))
    im12 = axes[1, 2].pcolormesh(lons, lats, pred_diff, cmap='RdBu_r', shading='auto',
                                 vmin=-max_abs_pred_diff, vmax=max_abs_pred_diff)
    axes[1, 2].set_title('Prediction Difference (Opt - Init)', fontsize=12)
    axes[1, 2].set_xlabel('Longitude (°E)', fontsize=11)
    axes[1, 2].set_ylabel('Latitude (°N)', fontsize=11)
    axes[1, 2].set_aspect('equal')
    plt.colorbar(im12, ax=axes[1, 2], label='Difference', fraction=0.025, pad=0.04)
    
    plt.tight_layout()
    return fig

print("✓ Prediction visualization function defined")

In [ ]:
def plot_gradient(gradient : torch.tensor, 
                  iteration : int,
                  ch : int, 
                  coords : dict) -> None :

    # Visualize gradients for channel at both time steps
    fig, axes = plt.subplots(1, 2, figsize=(18, 6))
    fig.suptitle(f'Standardized Gradients (Channel {ch}) - Iteration {iteration + 1}', fontsize=16, fontweight='bold')
    
    for t_idx in range(2):
        grad_t_ch = gradient[0, t_idx, ch].cpu().numpy()
        land_nan = np.where(ocean_mask_all[ch].cpu().numpy() == 0, np.nan, 1) # Already masked grad
        grad_t_ch = grad_t_ch * land_nan
        max_abs_grad = np.nanmax(np.abs(grad_t_ch))
        im = axes[t_idx].pcolormesh(coords['lon'], coords['lat'], grad_t_ch, 
                                            cmap='RdBu_r', shading='auto',
                                            vmin=-max_abs_grad, vmax=max_abs_grad)
        axes[t_idx].set_title(f'Time Step {t_idx}', fontsize=12)
        axes[t_idx].set_xlabel('Longitude (°E)', fontsize=11)
        axes[t_idx].set_ylabel('Latitude (°N)', fontsize=11)
        axes[t_idx].set_aspect('equal')
        plt.colorbar(im, ax=axes[t_idx], label='Standardized Gradient', fraction=0.025, pad=0.04)
    
    plt.tight_layout()
    return fig

In [ ]:
def plot_grad_histogram(gradient : torch.tensor, 
                        iteration : int) -> None :
    "Plot normalized histogram of gradients with statistics and log scale."
    
    fig, axes = plt.subplots(2, 2, figsize=(18, 12))
    fig.suptitle(f'Standardized Gradient Histograms - Iteration {iteration}', fontsize=16, fontweight='bold')
    
    var_names = ['SSH', 'THETAO', 'SO', 'UO', 'VO']
    colors = ['red', 'blue', 'green', 'orange', 'purple']
    
    for t_idx in range(2):
        # Collect all gradients for this timestep
        all_grads = {}
        stats_text = f"Timestep t={t_idx}\n" + "="*40 + "\n"
        
        for ch_idx, var_name in enumerate(var_names):
            if ch_idx == 0:  # SSH (2D)
                grad_data = gradient[0, t_idx, ch_idx].cpu().numpy()
            else:  # 3D variables
                grad_data = gradient[0, t_idx, ch_idx, :, :].cpu().numpy()
            
            # Flatten and remove NaN values (land points)
            grad_flat = grad_data.flatten()
            grad_flat = grad_flat[~np.isnan(grad_flat)]
            all_grads[var_name] = grad_flat
            
            # Statistics
            n_total = len(grad_flat)
            n_zero = np.sum(np.abs(grad_flat) < 1e-10)
            n_nonzero = n_total - n_zero
            pct_zero = (n_zero / n_total) * 100
            mean_val = np.mean(grad_flat)
            std_val = np.std(grad_flat)
            min_val = np.min(grad_flat)
            max_val = np.max(grad_flat)
            
            stats_text += f"{var_name:8s}: zero={pct_zero:5.1f}% | mean={mean_val:.2e} | std={std_val:.2e}\n"
            stats_text += f"          min={min_val:.2e} | max={max_val:.2e}\n"
        
        print(stats_text)
        
        # Plot 1: Linear scale (full range)
        ax_linear = axes[t_idx, 0]
        for ch_idx, var_name in enumerate(var_names):
            grad_flat = all_grads[var_name]
            # Compute histogram counts
            hist_vals, bin_edges = np.histogram(grad_flat, bins=100)
            bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
            ax_linear.plot(bin_centers, hist_vals, color=colors[ch_idx], 
                          linewidth=2.5, label=var_name, alpha=0.9)
            mean_val = np.mean(grad_flat)
            ax_linear.axvline(mean_val, color=colors[ch_idx], linestyle='--', 
                             linewidth=1.5, alpha=0.7)
        
        ax_linear.set_xlabel('Standardized Gradient Value', fontsize=11)
        ax_linear.set_ylabel('Count', fontsize=11)
        ax_linear.set_title(f'Timestep t={t_idx} - Linear Scale', fontsize=12)
        ax_linear.grid(True, alpha=0.3)
        ax_linear.legend(fontsize=9, loc='best')
        
        # Plot 2: Log scale
        ax_log = axes[t_idx, 1]
        for ch_idx, var_name in enumerate(var_names):
            grad_flat = all_grads[var_name]
            # Remove exact zeros for log scale
            grad_nonzero = grad_flat[np.abs(grad_flat) > 1e-15]
            if len(grad_nonzero) > 0:
                # Compute histogram counts
                hist_vals, bin_edges = np.histogram(grad_nonzero, bins=100)
                bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
                ax_log.plot(bin_centers, hist_vals, color=colors[ch_idx], 
                           linewidth=2.5, label=var_name, alpha=0.9)
                mean_val = np.mean(grad_nonzero)
                ax_log.axvline(mean_val, color=colors[ch_idx], linestyle='--', 
                              linewidth=1.5, alpha=0.7)
        
        ax_log.set_xlabel('Standardized Gradient Value', fontsize=11)
        ax_log.set_ylabel('Count (log scale)', fontsize=11)
        ax_log.set_title(f'Timestep t={t_idx} - Log Scale (non-zero only)', fontsize=12)
        ax_log.set_yscale('log')
        ax_log.grid(True, alpha=0.3)
        ax_log.legend(fontsize=9, loc='best')
    
    plt.tight_layout()
    return fig

In [ ]:
def plot_kernel_schedule(history, iteration):
    """Plot the kernel size schedule over iterations.
    
    Args:
        history: List of history dictionaries containing kernel_size
        iteration: Current iteration number
    """
    fig, ax = plt.subplots(1, 1, figsize=(14, 5))
    
    kernel_sizes = [h['kernel_size'] for h in history]
    iterations = list(range(len(kernel_sizes)))
    
    ax.plot(iterations, kernel_sizes, 'o-', linewidth=2.5, markersize=6, color='darkorange')
    ax.set_xlabel('Iteration', fontsize=12)
    ax.set_ylabel('Pooling Kernel Size', fontsize=12)
    ax.set_title(f'Multigrid Pooling Schedule - Iteration {iteration}', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)
    
    # Add annotations at kernel size changes
    prev_kernel = kernel_sizes[0]
    for i, k in enumerate(kernel_sizes):
        if k != prev_kernel:
            ax.axvline(i, color='red', linestyle='--', alpha=0.5, linewidth=1.5)
            ax.text(i, k, f'  k={k}', fontsize=10, color='red', 
                   verticalalignment='bottom', horizontalalignment='left')
            prev_kernel = k
    
    # Annotate initial kernel
    ax.text(0, kernel_sizes[0], f'  k={kernel_sizes[0]}', fontsize=10, color='darkorange',
           verticalalignment='bottom', horizontalalignment='left')
    
    plt.tight_layout()
    return fig

print("✓ Kernel schedule visualization function defined")

## 7. Initial State Analysis

In [ ]:
# Compute initial predictions
print("Computing initial predictions...")
with torch.no_grad():
    x0_in_std_init, y_hat_init_steps = forward(x0)
    y_hat_init = y_hat_init_steps[:, -1, :, :, :]

    initial_metrics = compute_rmse_metrics(y_hat_init, x0, x0_ref)
    mse_per_step_init, nmse_per_step_init, total_nloss_init, total_loss_init = compute_loss(y_hat_init_steps)
    
    nloss_input1_init = total_nloss_init[:, 0:5]
    # nloss_input2_init = total_nloss_init[:, 5:45]
    # nloss_input3_init = total_nloss_init[:, 45:85]
    
    nloss_ssh_init = total_nloss_init[:, 0:1]
    nloss_t_init = total_nloss_init[:, 1:2]
    nloss_s_init = total_nloss_init[:, 2:3]
    nloss_u_init = total_nloss_init[:, 3:4]
    nloss_v_init = total_nloss_init[:, 4:5]
    # nloss_t_init = torch.cat([total_nloss_init[:, 1:2], total_nloss_init[:, 5:15], total_nloss_init[:, 45:55]], dim=1)
    # nloss_s_init = torch.cat([total_nloss_init[:, 2:3], total_nloss_init[:, 15:25], total_nloss_init[:, 55:65]], dim=1)
    # nloss_u_init = torch.cat([total_nloss_init[:, 3:4], total_nloss_init[:, 25:35], total_nloss_init[:, 65:75]], dim=1)
    # nloss_v_init = torch.cat([total_nloss_init[:, 4:5], total_nloss_init[:, 35:45], total_nloss_init[:, 75:85]], dim=1)

print("\n" + "="*60)
print("INITIAL STATE ANALYSIS")
print("="*60)
print(f"Total Loss: {total_loss_init.sum().item():.6f}")
print(f"Per-step Losses: {[f'{v:.6f}' for v in nmse_per_step_init.sum(dim=2).squeeze(0).tolist()]}")
print(f"\nPrediction RMSE:")
for var, rmse in initial_metrics['pred'].items():
    if isinstance(rmse, list):
        rmse_str = [f"{r:.6e}" for r in rmse]
        print(f"  {var:10s}: {rmse_str}")
    else:
        print(f"  {var:10s}: {rmse:.6e}")
print(f"\nInitial Condition RMSE:")
for var, rmse in initial_metrics['ic'].items():
    if isinstance(rmse, list):
        rmse_str = [f"{r:.6e}" for r in rmse]
        print(f"  {var:10s}: {rmse_str}")
    else:
        print(f"  {var:10s}: {rmse:.6e}")
print("="*60)

In [ ]:
# Visualize initial predictions (SSH channel)
channel_to_plot = 0  # SSH channel
fig = plot_predictions(y_hat_init, y_hat_init, x0, x0_ref, coords, 0, channel_idx=channel_to_plot)
# plt.savefig(f"{CONFIG['output_dir']}/initial_prediction_ch{channel_to_plot}.png", dpi=150, bbox_inches='tight')
plt.show()

## 8. Optimization Loop

**Instructions:** Run this cell to start the optimization. 
It will automatically display visualizations every N iterations (configured in `plot_frequency`). 
You can stop it at any time using the stop button.

In [ ]:
# Initialize tracking variables
history = []
best_loss = float('inf')
best_y_hat = None
best_x0 = None
best_iteration = 0

# Get initial kernel size
initial_kernel = get_pooling_kernel_size(0, CONFIG)

# Store initial metrics
history.append({
    'loss': total_loss_init.sum().item(),
    'loss_steps': nmse_per_step_init.sum(dim=2).squeeze(0).detach().cpu().tolist(),
    'loss_input1': nloss_input1_init.sum().item(),
    # 'loss_input2': nloss_input2_init.sum().item(),
    # 'loss_input3': nloss_input3_init.sum().item(),
    'loss_ssh': nloss_ssh_init.sum().item(),
    'loss_t': nloss_t_init.sum().item(),
    'loss_s': nloss_s_init.sum().item(),
    'loss_u': nloss_u_init.sum().item(),
    'loss_v': nloss_v_init.sum().item(),
    'pred': initial_metrics['pred'],
    'ic': initial_metrics['ic'],
    'kernel_size': initial_kernel
})

print("\n" + "="*60)
print("STARTING OPTIMIZATION")
print("="*60)
print(f"You can stop at any time and inspect the results")
print(f"Plots will be displayed every {CONFIG['plot_frequency']} iterations")
if CONFIG.get('use_scheduled_pooling', False):
    schedule = CONFIG['pooling_schedule']
    print(f"Using Scheduled Pooling: {schedule['type']}")
    print(f"  Initial kernel: {initial_kernel}")
    print(f"  Final kernel: {schedule['final_kernel']}")
else:
    print(f"Using Fixed Pooling: kernel={CONFIG['pooling_kernel']}")
print("="*60)

In [ ]:
for iteration in range(CONFIG['num_iterations']):
    # Zero gradients
    if x0.grad is not None:
        x0.grad.zero_()
    
    # Compute current pooling kernel size based on schedule
    current_kernel = get_pooling_kernel_size(iteration, CONFIG)
    
    # Forward pass
    x0_in_std, y_hat_steps = forward(x0, CONFIG['observation_length'])
    y_hat = y_hat_steps[:, -1, :, :, :]
    
    # Compute loss
    mse_per_step, nmse_per_step, total_nloss, total_loss = compute_loss(y_hat_steps)
    loss_per_step = nmse_per_step.sum(dim=2)
    
    nloss_input1 = total_nloss[:, 0:5]
    # nloss_input2 = total_nloss[:, 5:45]
    # nloss_input3 = total_nloss[:, 45:85]
    
    nloss_ssh = total_nloss[:, 0:1]
    nloss_t = total_nloss[:, 1:2]
    nloss_s = total_nloss[:, 2:3]
    nloss_u = total_nloss[:, 3:4]
    nloss_v = total_nloss[:, 4:5]
    # nloss_t = torch.cat([total_nloss[:, 1:2], total_nloss[:, 5:15], total_nloss[:, 45:55]], dim=1)
    # nloss_s = torch.cat([total_nloss[:, 2:3], total_nloss[:, 15:25], total_nloss[:, 55:65]], dim=1)
    # nloss_u = torch.cat([total_nloss[:, 3:4], total_nloss[:, 25:35], total_nloss[:, 65:75]], dim=1)
    # nloss_v = torch.cat([total_nloss[:, 4:5], total_nloss[:, 35:45], total_nloss[:, 75:85]], dim=1)
        
    # Backward pass
    grads = torch.autograd.grad(total_loss.sum(), x0, create_graph=False)
    
    # Apply gradient filter with scheduled kernel
    filtered_grad = apply_gradient_filter(grads[0], x0, 
                                          filter_type=CONFIG['gradient_filter'],
                                          pooling_kernel=current_kernel)
    
    # Apply ocean mask to gradients and update
    with torch.no_grad():
        masked_grad = filtered_grad * ocean_mask_all.unsqueeze(0).unsqueeze(0)
        
        # Gradient descent update
        x0 = x0 - CONFIG['learning_rate'] * masked_grad
        
        x0.requires_grad = True
    
    # Compute metrics
    with torch.no_grad() :
        metrics = compute_rmse_metrics(y_hat, x0, x0_ref)
    
    # Store history (including kernel size and per-step losses)
    history.append({
        'loss': total_loss.sum().item(),
        'loss_steps': loss_per_step.squeeze(0).detach().cpu().tolist(),
        'loss_input1': nloss_input1.sum().item(),
        # 'loss_input2': nloss_input2.sum().item(),
        # 'loss_input3': nloss_input3.sum().item(),
        'loss_ssh': nloss_ssh.sum().item(),
        'loss_t': nloss_t.sum().item(),
        'loss_s': nloss_s.sum().item(),
        'loss_u': nloss_u.sum().item(),
        'loss_v': nloss_v.sum().item(),
        'pred': metrics['pred'],
        'ic': metrics['ic'],
        'kernel_size': current_kernel  # Track kernel size
    })
    
    # Update best
    if total_loss.sum().item() < best_loss:
        best_loss = total_loss.sum().item()
        best_y_hat = y_hat.detach().clone()
        best_x0 = x0.detach().clone()
        best_iteration = iteration + 1
    
    # Print and visualize
    if (iteration + 1) % CONFIG['plot_frequency'] == 0 or iteration == 0:
        clear_output(wait=True)
        
        print(f"\n{'='*60}")
        print(f"Iteration {iteration + 1}/{CONFIG['num_iterations']}")
        print(f"Gradient Filter: {CONFIG['gradient_filter']}")
        print(f"Pooling Kernel Size: {current_kernel}")
        print(f"{'='*60}")
        print(f"Total Loss: {total_loss.sum().item():.6f}")
        print(f"Per-step Losses: {[f'{v:.6f}' for v in loss_per_step.squeeze(0).tolist()]}")
        print(f"Best Loss: {best_loss:.6f} (iteration {best_iteration})")
        print(f"\nPrediction RMSE:")
        for var, rmse in metrics['pred'].items():
            if isinstance(rmse, list):
                rmse_str = [f"{r:.6e}" for r in rmse]
                print(f"  {var:10s}: {rmse_str}")
            else:
                print(f"  {var:10s}: {rmse:.6e}")
        print(f"\nInitial Condition RMSE:")
        for var, rmse in metrics['ic'].items():
            if isinstance(rmse, list):
                rmse_str = [f"{r:.6e}" for r in rmse]
                print(f"  {var:10s}: {rmse_str}")
            else:
                print(f"  {var:10s}: {rmse:.6e}")
        
        # Standardize gradient for visualization (per channel, per timestep)
        standardized_grad = masked_grad.clone()
        for t_idx in range(standardized_grad.shape[1]):
            for ch_idx in range(standardized_grad.shape[2]):
                grad_ch = standardized_grad[0, t_idx, ch_idx]
                # Get ocean points only (non-zero in mask)
                ocean_points = grad_ch[ocean_mask_all[ch_idx] > 0]
                if len(ocean_points) > 0:
                    mean_val = ocean_points.mean()
                    std_val = ocean_points.std()
                    if std_val > 1e-10:
                        # Standardize: (x - mean) / std
                        standardized_grad[0, t_idx, ch_idx] = (grad_ch - mean_val) / std_val
                        # Re-apply mask to ensure land points are zero
                        standardized_grad[0, t_idx, ch_idx] *= ocean_mask_all[ch_idx]
        
        print(f"\nGradient Statistics (after {CONFIG['gradient_filter']} filter, kernel={current_kernel}):")
        for ch_idx, var_name in enumerate(['SSH', 'THETAO', 'SO', 'UO', 'VO']):
            grad_ch = masked_grad[0, 1, ch_idx]  # Use last timestep
            ocean_points = grad_ch[ocean_mask_all[ch_idx] > 0]
            if len(ocean_points) > 0:
                print(f"  {var_name:8s}: mean={ocean_points.mean():.2e}, std={ocean_points.std():.2e}, "
                      f"min={ocean_points.min():.2e}, max={ocean_points.max():.2e}")
        
        # Plot gradients for variables
        for i in range(5):
            fig_grad = plot_gradient(standardized_grad, iteration + 1, ch=i, coords=coords)
        plt.show()
        
        fig_histo = plot_grad_histogram(standardized_grad, iteration + 1)
        plt.show()
        
        # Plot kernel schedule (if using scheduled pooling)
        if CONFIG.get('use_scheduled_pooling', False):
            fig_kernel = plot_kernel_schedule(history, iteration + 1)
            plt.show()
        
        # Plot RMSE evolution
        fig_rmse = plot_rmse_evolution(history, iteration + 1)
        plt.show()
        
        # Plot predictions
        fig_pred = plot_predictions(y_hat, y_hat_init, x0, x0_ref, coords, iteration + 1, channel_idx=1)
        plt.show()
    
    # Memory cleanup
    # del y_hat_steps
    del y_hat, mse_per_step, nmse_per_step, total_nloss, total_loss
    del nloss_ssh, nloss_t, nloss_s, nloss_u, nloss_v, grads
    gc.collect()
    torch.cuda.empty_cache()

print("\n" + "="*60)
print("OPTIMIZATION COMPLETE")
print("="*60)

In [ ]:
# Plot final RMSE evolution
fig = plot_rmse_evolution(history, len(history)-1)
plt.savefig(f"{CONFIG['output_dir']}/rmse_evolution_final.png", dpi=150, bbox_inches='tight')
plt.show()

# Plot final kernel schedule (if using scheduled pooling)
if CONFIG.get('use_scheduled_pooling', False):
    fig_kernel = plot_kernel_schedule(history, len(history)-1)
    plt.savefig(f"{CONFIG['output_dir']}/kernel_schedule_final.png", dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Plot best predictions for multiple channels
channels_to_plot = [0, 1, 2, 3, 4]  # SSH, thetao, so, uo, vo (surface)
channel_names = ['SSH', 'thetao_surface', 'so_surface', 'uo_surface', 'vo_surface']

for ch, name in zip(channels_to_plot, channel_names):
    print(f"\nPlotting channel {ch}: {name}")
    fig = plot_predictions(best_y_hat, y_hat_init, best_x0, x0_ref, coords, best_iteration, channel_idx=ch)
    plt.savefig(f"{CONFIG['output_dir']}/best_prediction_ch{ch}_{name}.png", 
               dpi=150, bbox_inches='tight')
    plt.show()

## 10. Save Results

In [ ]:
# Save history as DataFrame and JSON
import json

# Save history
history_df = pd.DataFrame([
    {
        'iteration': i,
        'loss': h['loss'],
        **{f'pred_{k}': v for k, v in h['pred'].items()},
        **{f'ic_{k}': v for k, v in h['ic'].items()}
    }
    for i, h in enumerate(history)
])

history_df.to_csv(f"{CONFIG['output_dir']}/optimization_history.csv", index=False)
print(f"✓ Saved optimization history to {CONFIG['output_dir']}/optimization_history.csv")

# Display first few rows
display(history_df.head(10))

In [ ]:
# Save configuration and results
results = {
    'config': CONFIG,
    'gradient_filter': CONFIG['gradient_filter'],
    'initial_loss': history[0]['loss'],
    'final_loss': history[-1]['loss'],
    'best_loss': best_loss,
    'best_iteration': best_iteration,
    'initial_metrics': history[0],
    'final_metrics': history[-1],
}

with open(f"{CONFIG['output_dir']}/results.json", 'w') as f:
    json.dump(results, f, indent=2, default=str)
print(f"✓ Saved results to {CONFIG['output_dir']}/results.json")

In [ ]:
# Save optimized initial conditions
with torch.no_grad():
    opt_x0_np = best_x0.cpu().numpy().squeeze(0)  # [T, 85, H, W]

ds_opt = xr.Dataset({
    'data': (['time', 'ch', 'lat', 'lon'], opt_x0_np)
}, coords={
    'time': coords['time'],
    # 'ch': np.arange(85),
    'ch': np.arange(5),
    'lat': coords['lat'],
    'lon': coords['lon']
})

ds_opt.attrs['description'] = 'Optimized initial conditions from gradient descent'
ds_opt.attrs['creation_date'] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
ds_opt.attrs['best_loss'] = float(best_loss)
ds_opt.attrs['best_iteration'] = best_iteration
ds_opt.attrs['learning_rate'] = CONFIG['learning_rate']
ds_opt.attrs['num_iterations'] = CONFIG['num_iterations']

encoding = {'data': {'zlib': True, 'complevel': 4}}
ds_opt.to_netcdf(f"{CONFIG['output_dir']}/optimized_initial_conditions.nc", encoding=encoding)
print(f"✓ Saved optimized initial conditions to {CONFIG['output_dir']}/optimized_initial_conditions.nc")

In [ ]:
# Save best predictions
with torch.no_grad():
    best_pred_np = best_y_hat.cpu().numpy()  # [1, 85, H, W]

ds_pred = xr.Dataset({
    'data': (['time', 'ch', 'lat', 'lon'], best_pred_np)
}, coords={
    'time': [coords['target_time']],
    # 'ch': np.arange(85),
    'ch': np.arange(5),
    'lat': coords['lat'],
    'lon': coords['lon']
})

ds_pred.attrs['description'] = 'Best forecast predictions from optimized initial conditions'
ds_pred.attrs['creation_date'] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
ds_pred.attrs['best_loss'] = float(best_loss)
ds_pred.attrs['best_iteration'] = best_iteration

encoding = {'data': {'zlib': True, 'complevel': 4}}
ds_pred.to_netcdf(f"{CONFIG['output_dir']}/best_predictions.nc", encoding=encoding)
print(f"✓ Saved best predictions to {CONFIG['output_dir']}/best_predictions.nc")

## 11. Anaylsis for Forecast over observation series

From optimized initial condition, see the forecast performance beyond the observation series. </br>
[!] In this notebook, parameter `forecast_horizon` is used in two ways: **observation series** and **forecast window**.

In [ ]:
x0_t = y_hat_steps[:, -3:-1, :, :, :] # [1, 2, 5, H, W] 
print(f"Shape of y_hat_steps: {y_hat_steps.shape}")
print(f"Shape of x0_t: {x0_t.shape}")

_, best_output_steps = forward(best_x0, CONFIG['observation_length'])
x0_t = best_output_steps[:, -3:-1, :, :, :] # [1, 2, 5, H, W] 
_, forecast_output_steps = forward(x0_t, CONFIG['forecast_horizon'])
print(f"Shape of forecast_output_steps: {forecast_output_steps.shape}")

In [ ]:
# Redefine target sequence and target for forecast evaluation
forecast_target_start_idx = CONFIG['sample_idx'] + CONFIG['sequence_length'] + CONFIG['forecast_horizon']
forecast_target_end_idx = forecast_target_start_idx + CONFIG['forecast_horizon']
forecast_target_sequence = dataset.isel(time=slice(forecast_target_start_idx, forecast_target_end_idx))
forecast_target_time_idx = forecast_target_end_idx - 1
forecast_target = dataset.isel(time=forecast_target_time_idx)

# Store coordinates
coords = {
    'time': input_sequence['time'].values,
    'target_times': forecast_target_sequence['time'].values,
    'target_time': forecast_target['time'].values,
    'lat': input_sequence['lat'].values,
    'lon': input_sequence['lon'].values
}

print(f"Input sequence shape: {input_sequence['data'].shape}")
print(f"Target sequence shape: {forecast_target_sequence['data'].shape}")
print(f"Final target shape: {forecast_target['data'].shape}")
print(f"Final target time: {coords['target_time']}")

In [ ]:
# # Add noise to truth to make more realistic observations.
# # e ~ N(0, var_channel), with its own control variable for forecast target
# forecast_noise_scale = 1.0
# forecast_noise_seed = 123
# rng_forecast = np.random.default_rng(forecast_noise_seed)

# # Fallback helper if the earlier cell was not executed
# if 'add_channelwise_gaussian_noise' not in globals():
#     def add_channelwise_gaussian_noise(ds, var_name='data', scale=1.0):
#         da = ds[var_name]
#         var_by_channel = da.var(dim=('time', 'lat', 'lon'), skipna=True)
#         std_by_channel = np.sqrt(var_by_channel.values + 1e-12) * scale

#         noise = rng_forecast.normal(
#             loc=0.0,
#             scale=std_by_channel[None, :, None, None],
#             size=da.shape
#         )

#         ds_noisy = ds.copy()
#         ds_noisy[var_name] = da + xr.DataArray(
#             noise,
#             dims=da.dims,
#             coords=da.coords
#         )
#         return ds_noisy, var_by_channel

# # Rebind RNG used by the shared helper from earlier cells
# rng = rng_forecast

# # Use the same formula: e ~ N(0, var_channel)
# forecast_target_sequence, forecast_target_var_by_channel = add_channelwise_gaussian_noise(
#     forecast_target_sequence,
#     scale=forecast_noise_scale
# )

# print('Added channel-wise Gaussian noise to forecast_target_sequence: e ~ N(0, var_channel)')
# print(f'Forecast noise scale: {forecast_noise_scale}, seed: {forecast_noise_seed}')
# print('Forecast target variance by channel:')
# print(forecast_target_var_by_channel.values)

In [ ]:
# Compare forecast_output_steps with forecast_target_sequence
with torch.no_grad():
    # Prediction tensor: [1, T_pred, 5, H, W]
    pred_steps = forecast_output_steps
    
    # Target tensor from forecast_target_sequence: [1, T_tgt, 5, H, W]
    forecast_target_np = np.nan_to_num(
        forecast_target_sequence['data'].values[:, 0:5, :, :],
        nan=0.0
    )
    target_steps = torch.from_numpy(forecast_target_np.copy()).float().unsqueeze(0).to(pred_steps.device)
    
    # Align to common number of time steps
    n_steps = min(pred_steps.shape[1], target_steps.shape[1])
    pred_steps = pred_steps[:, :n_steps, :, :, :]
    target_steps = target_steps[:, :n_steps, :, :, :]
    
    # Ocean-mask-aware RMSE per timestep/channel
    # rmse_per_step_ch shape: [T, 5]
    se = (pred_steps - target_steps) ** 2
    se_masked = se * ocean_mask_all.unsqueeze(0).unsqueeze(0)
    n_ocean_points = ocean_mask_all.unsqueeze(0).sum(dim=(2, 3)).unsqueeze(1)  # [1, 1, 5]
    mse_per_step_ch = se_masked.sum(dim=(3, 4)) / (n_ocean_points + 1e-10)
    rmse_per_step_ch = torch.sqrt(mse_per_step_ch).squeeze(0).cpu().numpy()
    
    channel_names = ['SSH', 'thetao', 'so', 'uo', 'vo']
    
    
    
    
# Print table
print(f"Compared {n_steps} steps (pred={forecast_output_steps.shape[1]}, target={target_steps.shape[1]}).")
print("\nRMSE per time step and channel:")
header = "step " + " ".join([f"{name:>12s}" for name in channel_names])
print(header)
for t in range(n_steps):
    row_vals = " ".join([f"{rmse_per_step_ch[t, c]:12.6e}" for c in range(5)])
    print(f"{t+1:>4d} {row_vals}")

In [ ]:
# Plot RMSE development
steps = np.arange(1, n_steps + 1)
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Left: per-channel RMSE evolution
for c, name in enumerate(channel_names):
    axes[0].plot(steps, rmse_per_step_ch[:, c], marker='o', linewidth=2, label=name)
axes[0].set_xlabel('Forecast Step')
axes[0].set_ylabel('RMSE')
axes[0].set_title('RMSE Development by Channel')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

# Right: mean RMSE over 5 channels
mean_rmse = rmse_per_step_ch.mean(axis=1)
axes[1].plot(steps, mean_rmse, marker='o', linewidth=2.5, color='black')
axes[1].set_xlabel('Forecast Step')
axes[1].set_ylabel('RMSE')
axes[1].set_title('Mean RMSE Across 5 Channels')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Same RMSE calculation and plotting for reference result
with torch.no_grad():
    # Double forecasting : possibel only length of observation serise is same to length of forecast horizon
    _, forecast_t_ref = forward(x0_ref, CONFIG['observation_length'])
    x_t_ref = forecast_t_ref[:, -3:-1, :, :, :] # [1, 2, 5, H, W]
    _, forecast_output_steps_ref = forward(x_t_ref, CONFIG['forecast_horizon'])

    forecast_target_np = np.nan_to_num(
        forecast_target_sequence['data'].values[:, 0:5, :, :],
        nan=0.0
    )
    target_steps_ref = torch.from_numpy(forecast_target_np.copy()).float().unsqueeze(0).to(forecast_output_steps_ref.device)
    
    n_steps_ref = min(forecast_output_steps_ref.shape[1], target_steps_ref.shape[1])
    pred_steps_ref = forecast_output_steps_ref[:, :n_steps_ref, :, :, :]
    target_steps_ref = target_steps_ref[:, :n_steps_ref, :, :, :]
    
    se_ref = (pred_steps_ref - target_steps_ref) ** 2
    se_masked_ref = se_ref * ocean_mask_all.unsqueeze(0).unsqueeze(0)
    n_ocean_points_ref = ocean_mask_all.unsqueeze(0).sum(dim=(2, 3)).unsqueeze(1)
    mse_per_step_ch_ref = se_masked_ref.sum(dim=(3, 4)) / (n_ocean_points_ref + 1e-10)
    rmse_per_step_ch_ref = torch.sqrt(mse_per_step_ch_ref).squeeze(0).cpu().numpy()
    
    channel_names_ref = ['SSH', 'thetao', 'so', 'uo', 'vo']
    
    print(f"Reference run compared {n_steps_ref} steps (pred={forecast_output_steps_ref.shape[1]}, target={target_steps_ref.shape[1]}).")
    print("\nReference RMSE per time step and channel:")
    header_ref = "step " + " ".join([f"{name:>12s}" for name in channel_names_ref])
    print(header_ref)
    for t in range(n_steps_ref):
        row_vals_ref = " ".join([f"{rmse_per_step_ch_ref[t, c]:12.6e}" for c in range(5)])
        print(f"{t+1:>4d} {row_vals_ref}")
    
    steps_ref = np.arange(1, n_steps_ref + 1)
    fig_ref, axes_ref = plt.subplots(1, 2, figsize=(16, 5))
    
    for c, name in enumerate(channel_names_ref):
        axes_ref[0].plot(steps_ref, rmse_per_step_ch_ref[:, c], marker='o', linewidth=2, label=name)
    axes_ref[0].set_xlabel('Forecast Step')
    axes_ref[0].set_ylabel('RMSE')
    axes_ref[0].set_title('Reference RMSE Development by Channel')
    axes_ref[0].grid(True, alpha=0.3)
    axes_ref[0].legend()
    
    mean_rmse_ref = rmse_per_step_ch_ref.mean(axis=1)
    axes_ref[1].plot(steps_ref, mean_rmse_ref, marker='o', linewidth=2.5, color='black')
    axes_ref[1].set_xlabel('Forecast Step')
    axes_ref[1].set_ylabel('RMSE')
    axes_ref[1].set_title('Reference Mean RMSE Across 5 Channels')
    axes_ref[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Overlay RMSE development by channel: optimized vs reference
if 'rmse_per_step_ch' not in globals() or 'rmse_per_step_ch_ref' not in globals():
    raise RuntimeError('Run RMSE cells first to create rmse_per_step_ch and rmse_per_step_ch_ref.')

rmse_opt = np.asarray(rmse_per_step_ch)
rmse_ref = np.asarray(rmse_per_step_ch_ref)

# Align horizons for fair comparison
n_steps_cmp = min(rmse_opt.shape[0], rmse_ref.shape[0])
rmse_opt_cmp = rmse_opt[:n_steps_cmp, :]
rmse_ref_cmp = rmse_ref[:n_steps_cmp, :]

# Standard deviation from the last reference state (target at final forecast step)
forecast_target_np = np.nan_to_num(
    forecast_target_sequence['data'].values[:, 0:5, :, :],
    nan=0.0
)
last_ref_state = torch.from_numpy(forecast_target_np[-1].copy()).float().to(ocean_mask_all.device)  # [5, H, W]

std_by_channel = []
for ch in range(5):
    ocean_points = last_ref_state[ch][ocean_mask_all[ch] > 0]
    std_val = ocean_points.std().item() if ocean_points.numel() > 0 else float('nan')
    std_by_channel.append(std_val)
std_by_channel = np.array(std_by_channel)

channel_names = ['SSH', 'thetao', 'so', 'uo', 'vo']
steps = np.arange(1, n_steps_cmp + 1)

print(f'Compared {n_steps_cmp} common steps.')
print('Channel-wise standard deviation from last reference state:')
for ch, name in enumerate(channel_names):
    print(f'  {name:>6s}: std={std_by_channel[ch]:.6e}')

fig, axes = plt.subplots(3, 2, figsize=(16, 13))
axes = axes.flatten()

for ch, name in enumerate(channel_names):
    ax = axes[ch]
    ax.plot(steps, rmse_opt_cmp[:, ch], marker='o', linewidth=2.2, label='Optimized RMSE')
    ax.plot(steps, rmse_ref_cmp[:, ch], marker='s', linewidth=2.2, label='Reference RMSE')
    ax.axhline(std_by_channel[ch], color='gray', linestyle='--', linewidth=1.8,
               label=f'Std(last ref state)={std_by_channel[ch]:.2e}')
    ax.set_title(f'{name} RMSE Development')
    ax.set_xlabel('Forecast Step')
    ax.set_ylabel('RMSE')
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=9)

# Hide the unused 6th panel
axes[-1].axis('off')

plt.tight_layout()
compare_plot_path = f"{CONFIG['output_dir']}/rmse_comparison_optimized_vs_reference.png"
plt.savefig(compare_plot_path, dpi=150, bbox_inches='tight')
print(f"Saved comparison plot to {compare_plot_path}")
plt.show()

In [ ]:
# PSD comparison: initial condition and forecast ocean states (wavenumber domain)
required_vars = [
    'x0_ref', 'best_x0', 'ocean_mask_all', 'coords',
    'forecast_output_steps', 'forecast_output_steps_ref', 'forecast_target_sequence', 'CONFIG'
 ]
missing = [v for v in required_vars if v not in globals()]
if missing:
    raise RuntimeError(f"Missing variables: {missing}. Run previous forecast cells first.")

channel_names = ['SSH', 'thetao', 'so', 'uo', 'vo']
mask_np = ocean_mask_all.detach().cpu().numpy().astype(bool)  # [5, H, W]

# Grid spacing in degrees (GLORYS12 is reinterploate in 1/4 degree)
dlon = float(np.abs(np.median(np.diff(coords['lon']))))
dlat = float(np.abs(np.median(np.diff(coords['lat']))))

grid_resolution_deg = 0.25
dlon = 0.25
dlat = 0.25

# Nyquist frequency in cycle/degree: f_N = 1/(2*dx)
f_N = 1.0 / (2.0 * grid_resolution_deg)  # 2.0 cycle/degree for 1/4 degree grid
print(f"Using Nyquist frequency f_N={f_N:.3f} cycle/degree")


def isotropic_psd(field2d, ocean_mask2d, dlon=0.25, dlat=0.25, n_bins=140, f_nyquist=2.0):
    """Compute isotropic 2D PSD binned by radial wavenumber on a logarithmic grid.

    Frequencies above f_nyquist are removed.
    """
    f = np.asarray(field2d, dtype=np.float64)
    m = np.asarray(ocean_mask2d, dtype=bool)

    if f.shape != m.shape:
        raise ValueError(f"Shape mismatch: field={f.shape}, mask={m.shape}")

    # Ensure requested cutoff does not exceed theoretical Nyquist from grid spacing
    f_nyquist_theoretical = min(1.0 / (2.0 * dlon), 1.0 / (2.0 * dlat))
    f_cut = min(float(f_nyquist), float(f_nyquist_theoretical))

    # Remove mean over selected points only, then set all other points to zero
    ocean_vals = f[m]
    ocean_mean = ocean_vals.mean() if ocean_vals.size > 0 else 0.0
    f_centered = np.where(m, f - ocean_mean, 0.0)

    # Apply 2D Hann window to reduce spectral leakage
    wy = np.hanning(f.shape[0])
    wx = np.hanning(f.shape[1])
    window2d = np.outer(wy, wx)
    f_windowed = f_centered * window2d

    # 2D FFT power spectrum
    F = np.fft.fft2(f_windowed)
    P2 = (np.abs(F) ** 2) / (f.shape[0] * f.shape[1])

    # Wavenumber grid (cycles per degree)
    ky = np.fft.fftfreq(f.shape[0], d=dlat)
    kx = np.fft.fftfreq(f.shape[1], d=dlon)
    KX, KY = np.meshgrid(kx, ky)
    KR = np.sqrt(KX ** 2 + KY ** 2)

    kr_flat = KR.ravel()
    p_flat = P2.ravel()

    # Remove zero-frequency component and all components above Nyquist cutoff
    valid = (kr_flat > 0) & (kr_flat <= f_cut)
    kr_flat = kr_flat[valid]
    p_flat = p_flat[valid]

    if kr_flat.size == 0:
        return np.array([]), np.array([])

    # Logarithmic wavenumber bins
    k_min = kr_flat.min()
    k_max = kr_flat.max()
    if k_max <= k_min:
        return np.array([]), np.array([])

    bins = np.logspace(np.log10(k_min), np.log10(k_max), n_bins + 1)
    which_bin = np.digitize(kr_flat, bins) - 1

    psd_bin = np.full(n_bins, np.nan, dtype=np.float64)
    k_bin = np.sqrt(bins[:-1] * bins[1:])  # geometric center for log bins

    for b in range(n_bins):
        in_bin = which_bin == b
        if np.any(in_bin):
            psd_bin[b] = p_flat[in_bin].mean()

    keep = np.isfinite(psd_bin) & (psd_bin > 0) & (k_bin > 0)

    return k_bin[keep], psd_bin[keep]


In [ ]:
def build_regional_masks(mask_ocean_ch, coords_dict, variance_source, high_q=0.80, low_q=0.20):
    """Create global / high-variance / low-variance masks for PSD analysis.

    High-variance region combines:
      1) dynamic high-variance quantile from data
      2) canonical energetic regions (Gulf Stream, Kuroshio, Antarctic circumpolar band, Agulhas)

    Returns:
      regional_masks: dict(region -> [C, H, W] bool)
      variance_map: [H, W] float, mean channel variance over time
    """
    lon = np.asarray(coords_dict['lon'])
    lat = np.asarray(coords_dict['lat'])
    lon2d, lat2d = np.meshgrid(lon, lat)

    # Time-variance map from reference truth over forecast period
    var_src = np.nan_to_num(variance_source['data'].values[:, 0:5, :, :], nan=0.0)  # [T, 5, H, W]
    var_map_ch = np.var(var_src, axis=0)  # [5, H, W]
    variance_map = var_map_ch.mean(axis=0)  # [H, W]

    regional_masks = {'global': [], 'high_var': [], 'low_var': []}

    # Geographical energetic regions (approximate boxes)
    gulf_stream_box = (lat2d >= 30.0) & (lat2d <= 48.0) & (lon2d >= 280.0) & (lon2d <= 330.0)
    kuroshio_box = (lat2d >= 25.0) & (lat2d <= 40.0) & (lon2d >= 130.0) & (lon2d <= 165.0)
    antarctic_band = (lat2d >= -65.0) & (lat2d <= -45.0)
    agulhas_box = (lat2d >= -45.0) & (lat2d <= -30.0) & (lon2d >= 10.0) & (lon2d <= 45.0)
    geo_high_var = gulf_stream_box | kuroshio_box | antarctic_band | agulhas_box

    for ch in range(mask_ocean_ch.shape[0]):
        ocean_ch = mask_ocean_ch[ch]
        var_vals = variance_map[ocean_ch]

        if var_vals.size == 0:
            high_dyn = np.zeros_like(ocean_ch, dtype=bool)
            low_dyn = np.zeros_like(ocean_ch, dtype=bool)
        else:
            q_hi = np.quantile(var_vals, high_q)
            q_lo = np.quantile(var_vals, low_q)
            high_dyn = (variance_map >= q_hi) & ocean_ch
            low_dyn = (variance_map <= q_lo) & ocean_ch

        high_mask = (high_dyn | geo_high_var) & ocean_ch
        low_mask = low_dyn & (~high_mask) & ocean_ch

        regional_masks['global'].append(ocean_ch)
        regional_masks['high_var'].append(high_mask)
        regional_masks['low_var'].append(low_mask)

    regional_masks = {k: np.stack(v, axis=0) for k, v in regional_masks.items()}
    return regional_masks, variance_map


regional_masks, variance_map = build_regional_masks(mask_np, coords, forecast_target_sequence)

# Basic diagnostics for regional coverage
for region_name, rmask in regional_masks.items():
    frac_by_ch = []
    for ch in range(rmask.shape[0]):
        ocean_count = mask_np[ch].sum()
        region_count = rmask[ch].sum()
        frac = (region_count / ocean_count * 100.0) if ocean_count > 0 else np.nan
        frac_by_ch.append(frac)
    frac_txt = ', '.join([f"{channel_names[i]}={frac_by_ch[i]:.1f}%" for i in range(5)])
    print(f"Region '{region_name}' coverage: {frac_txt}")

In [ ]:
# Visual check: ocean mask vs high/low variance regional masks
required_vars = ['mask_np', 'regional_masks', 'channel_names', 'coords']
missing = [v for v in required_vars if v not in globals()]
if missing:
    raise RuntimeError(f"Missing variables: {missing}. Run the regional mask cell first.")

lon = np.asarray(coords['lon'])
lat = np.asarray(coords['lat'])

high_mask = regional_masks['high_var']
low_mask = regional_masks['low_var']

fig, axes = plt.subplots(5, 4, figsize=(18, 20), constrained_layout=True)
fig.suptitle('Mask Validation: Ocean vs High/Low Variance Regions', fontsize=16, fontweight='bold')

for ch, name in enumerate(channel_names):
    ocean = mask_np[ch].astype(bool)
    high = high_mask[ch].astype(bool)
    low = low_mask[ch].astype(bool)

    # Sanity checks
    overlap = high & low
    outside_ocean_high = high & (~ocean)
    outside_ocean_low = low & (~ocean)

    n_ocean = int(ocean.sum())
    n_high = int(high.sum())
    n_low = int(low.sum())
    n_overlap = int(overlap.sum())
    n_out_high = int(outside_ocean_high.sum())
    n_out_low = int(outside_ocean_low.sum())

    high_pct = (100.0 * n_high / n_ocean) if n_ocean > 0 else np.nan
    low_pct = (100.0 * n_low / n_ocean) if n_ocean > 0 else np.nan

    print(
        f"[{name}] ocean={n_ocean:,d}, high={n_high:,d} ({high_pct:.2f}%), "
        f"low={n_low:,d} ({low_pct:.2f}%), overlap={n_overlap:,d}, "
        f"high_outside_ocean={n_out_high:,d}, low_outside_ocean={n_out_low:,d}"
    )

    # Col 1: ocean mask
    ax = axes[ch, 0]
    ax.pcolormesh(lon, lat, ocean.astype(float), cmap='Blues', shading='auto', vmin=0, vmax=1)
    ax.set_title(f"{name}: Ocean Mask")
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_aspect('equal')

    # Col 2: high variance mask
    ax = axes[ch, 1]
    ax.pcolormesh(lon, lat, high.astype(float), cmap='Reds', shading='auto', vmin=0, vmax=1)
    ax.set_title(f"{name}: High Variance")
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_aspect('equal')

    # Col 3: low variance mask
    ax = axes[ch, 2]
    ax.pcolormesh(lon, lat, low.astype(float), cmap='Greens', shading='auto', vmin=0, vmax=1)
    ax.set_title(f"{name}: Low Variance")
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_aspect('equal')

    # Col 4: overlay for quick validation
    # 0=background, 1=ocean-only, 2=high, 3=low, 4=overlap(high&low)
    overlay = np.zeros_like(ocean, dtype=np.int32)
    overlay[ocean] = 1
    overlay[high] = 2
    overlay[low] = 3
    overlay[overlap] = 4

    cmap = mcolors.ListedColormap(['black', '#d9d9d9', '#d7191c', '#1a9641', '#ff00ff'])
    bounds = [-0.5, 0.5, 1.5, 2.5, 3.5, 4.5]
    norm = mcolors.BoundaryNorm(bounds, cmap.N)

    ax = axes[ch, 3]
    im = ax.pcolormesh(lon, lat, overlay, cmap=cmap, norm=norm, shading='auto')
    ax.set_title(f"{name}: Overlay")
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_aspect('equal')

# Single colorbar for overlay categories
cbar = fig.colorbar(im, ax=axes[:, 3], fraction=0.02, pad=0.02, ticks=[0, 1, 2, 3, 4])
cbar.ax.set_yticklabels(['background', 'ocean', 'high', 'low', 'overlap'])

plt.show()

In [ ]:
# Plot 1: PSD of initial conditions by region

ic_time_idx = min(1, x0_ref.shape[1] - 1)
ic_original = x0_ref[0, ic_time_idx].detach().cpu().numpy()     # [5, H, W]
ic_optimized = best_x0[0, ic_time_idx].detach().cpu().numpy()   # [5, H, W]

for region_name, region_mask in regional_masks.items():
    fig_ic, axes_ic = plt.subplots(3, 2, figsize=(16, 13))
    axes_ic = axes_ic.flatten()

    for ch, name in enumerate(channel_names):
        k_org, p_org = isotropic_psd(ic_original[ch], region_mask[ch], dlon=dlon, dlat=dlat, f_nyquist=f_N)
        k_opt, p_opt = isotropic_psd(ic_optimized[ch], region_mask[ch], dlon=dlon, dlat=dlat, f_nyquist=f_N)

        ax = axes_ic[ch]
        ax.loglog(k_org, p_org, linewidth=2.0, label='Reference IC')
        ax.loglog(k_opt, p_opt, linewidth=2.0, label='Optimized IC')
        ax.axvline(f_N, color='gray', linestyle='--', linewidth=1.5, label=f'Nyquist={f_N:.2f}')
        ax.set_title(f'{name}: PSD of Initial Condition ({region_name})')
        ax.set_xlabel('Wavenumber (cycle/degree)')
        ax.set_ylabel('PSD')
        ax.grid(True, which='both', alpha=0.3)
        ax.legend(fontsize=9)

    axes_ic[-1].axis('off')
    plt.tight_layout()
    ic_psd_path = f"{CONFIG['output_dir']}/psd_reference_vs_optimized_IC_{region_name}.png"
    plt.savefig(ic_psd_path, dpi=150, bbox_inches='tight')
    print(f"Saved IC PSD comparison ({region_name}) to {ic_psd_path}")
    plt.show()

In [ ]:
# Plot 2: PSD among reference, control, optimized forecasts by region

# Compare at the same forecast step (last common step)
n_fc_common = min(
    forecast_output_steps.shape[1],
    forecast_output_steps_ref.shape[1],
    forecast_target_sequence['data'].shape[0]
 )
if n_fc_common < 1:
    raise RuntimeError("No common forecast step available for PSD comparison.")

fc_idx = n_fc_common - 1
last_reference = np.nan_to_num(forecast_target_sequence['data'].values[fc_idx, 0:5, :, :], nan=0.0)
last_control = forecast_output_steps_ref[0, fc_idx].detach().cpu().numpy()
last_optimized = forecast_output_steps[0, fc_idx].detach().cpu().numpy()

for region_name, region_mask in regional_masks.items():
    fig_fc, axes_fc = plt.subplots(3, 2, figsize=(16, 13))
    axes_fc = axes_fc.flatten()

    for ch, name in enumerate(channel_names):
        k_ref, p_ref = isotropic_psd(last_reference[ch], region_mask[ch], dlon=dlon, dlat=dlat, f_nyquist=f_N)
        k_ctl, p_ctl = isotropic_psd(last_control[ch], region_mask[ch], dlon=dlon, dlat=dlat, f_nyquist=f_N)
        k_opt, p_opt = isotropic_psd(last_optimized[ch], region_mask[ch], dlon=dlon, dlat=dlat, f_nyquist=f_N)

        ax = axes_fc[ch]
        ax.loglog(k_ref, p_ref, linewidth=2.0, label='Reference (Target)')
        ax.loglog(k_opt, p_opt, linewidth=2.0, label='Optimized Forecast')
        ax.loglog(k_ctl, p_ctl, linewidth=2.0, label='Control Forecast (Original IC)')
        ax.axvline(f_N, color='gray', linestyle='--', linewidth=1.5, label=f'Nyquist={f_N:.2f}')
        ax.set_title(f'{name}: PSD of Ocean State ({region_name}, Step {fc_idx + 1})')
        ax.set_xlabel('Wavenumber (cycle/degree)')
        ax.set_ylabel('PSD')
        ax.grid(True, which='both', alpha=0.3)
        ax.legend(fontsize=8)

    axes_fc[-1].axis('off')
    plt.tight_layout()
    fc_psd_path = f"{CONFIG['output_dir']}/psd_reference_control_optimized_forecast_{region_name}.png"
    plt.savefig(fc_psd_path, dpi=150, bbox_inches='tight')
    print(f"Saved forecast PSD comparison ({region_name}) to {fc_psd_path}")
    plt.show()

print(f"Forecast PSD compared at common step: {fc_idx + 1} / {n_fc_common}")

## 12. Save forecate ocean state

In [ ]:
# Save ocean state in netcdf file format

forecast_output_steps_np = forecast_output_steps.squeeze().squeeze().cpu().numpy()  # [T_forecast, 5, H, W]

# Use target_times as coordinate for all forecast_horizon steps
time_coords = np.asarray(coords['target_times'])
if time_coords.shape[0] != forecast_output_steps_np.shape[0]:
    raise ValueError(
        f"Time coordinate length ({time_coords.shape[0]}) does not match forecast steps ({forecast_output_steps_np.shape[0]})."
    )

ds_forecast = xr.Dataset({
    'data': (['time', 'ch', 'lat', 'lon'], forecast_output_steps_np)
}, coords={
    'time': time_coords,
    # 'ch': np.arange(85),
    'ch': np.arange(5),
    'lat': coords['lat'],
    'lon': coords['lon']
})

ds_forecast.attrs['description'] = 'Forecast from optimized initial conditions'
ds_forecast.attrs['creation_date'] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
ds_forecast.attrs['observation_window'] = CONFIG['forecast_horizon']
ds_forecast.attrs['forecast_window'] = CONFIG['forecast_horizon']
ds_forecast.attrs['reference'] = 'GLONET forecast from GLORYS12 reanalysis 1/4° initial condition'

rmse_last = np.asarray(rmse_per_step_ch)[-1]
ds_forecast.attrs['last_rmse_SSH'] = float(rmse_last[0])
ds_forecast.attrs['last_rmse_thetao'] = float(rmse_last[1])
ds_forecast.attrs['last_rmse_so'] = float(rmse_last[2])
ds_forecast.attrs['last_rmse_uo'] = float(rmse_last[3])
ds_forecast.attrs['last_rmse_vo'] = float(rmse_last[4])

encoding = {'data': {'zlib': True, 'complevel': 4}}
ds_forecast.to_netcdf(f"{CONFIG['output_dir']}/forecast_from_optimized_IC.nc", encoding=encoding)
print('✓ Added last RMSE attributes and updated forecast_from_optimized_IC.nc')

In [ ]:
# Save RMSE-by-step comparison to JSON
import json

if 'rmse_per_step_ch' not in globals() or 'rmse_per_step_ch_ref' not in globals():
    raise RuntimeError('Run RMSE cells first to create rmse_per_step_ch and rmse_per_step_ch_ref.')

rmse_opt = np.asarray(rmse_per_step_ch)
rmse_ref = np.asarray(rmse_per_step_ch_ref)

# Align to a common horizon for direct comparison
n_steps_cmp = min(rmse_opt.shape[0], rmse_ref.shape[0])
rmse_opt_cmp = rmse_opt[:n_steps_cmp, :]
rmse_ref_cmp = rmse_ref[:n_steps_cmp, :]
rmse_diff_cmp = rmse_opt_cmp - rmse_ref_cmp

channel_names = ['SSH', 'thetao', 'so', 'uo', 'vo']
steps = list(range(1, n_steps_cmp + 1))

rmse_payload = {
    'channel_names': channel_names,
    'steps': steps,
    'optimized_rmse_by_step': rmse_opt_cmp.tolist(),
    'reference_rmse_by_step': rmse_ref_cmp.tolist(),
    'difference_optimized_minus_reference': rmse_diff_cmp.tolist(),
    'mean_rmse': {
        'optimized': rmse_opt_cmp.mean(axis=1).tolist(),
        'reference': rmse_ref_cmp.mean(axis=1).tolist()
    }
}

rmse_json_path = f"{CONFIG['output_dir']}/rmse_step_comparison.json"
with open(rmse_json_path, 'w') as f:
    json.dump(rmse_payload, f, indent=2)

print(f"Saved RMSE step JSON to {rmse_json_path}")
print(f"Saved {n_steps_cmp} forecast steps across {len(channel_names)} channels")